# Bygge grafen

Målet er å skape en graf som
- Er fri for skala, og
- som også holder våre kriminelle.
Det vil i praksis si å lese inn grafen, og så legge til nodene (og deres relasjoner) slik vi har forberedt.

For at våre kriminelle skal gli naturlig inn, må de gis relasjoner til de eksisterende på en "naturlig" måte.  Det vil si at hver node må få (et antall) relasjoner tli andre noder som "ligner".  I praksis betyr det å ha større sjande for å knytte seg til noder som alerede har mange kanter.

Nå har (noen) av våre kriminelle trolig flere kanter enn medianen i datasettet, men det kan vi se på senere.

Eneste måten jeg har funnet for å legge inn ny noder på en "normal" måte, er å lage et sett av alle noder, hvor hver node er i settet like mange ganger som det har kanter.  Når én node nå trekkes fra settet vil det sannsynligheten for å trekke en node reflektere nodens sentralitet.





In [11]:
# Laste inn datasettet
# La oss lage en graf
import networkx as nx
import gzip

EG = nx.DiGraph()
# husk at gzip åpner i 'b'
with gzip.open("data/email.edgelist.txt.gz", "rt") as fd:
    for linje in fd:
         link = linje.split()
         EG.add_edge(int(link[0]), int(link[1]))
    #
#

# Merke nodene
nx.set_node_attributes(EG, True, name="Epost")

print(f"Antall noder: {EG.number_of_nodes()}")
print(f"Andtall kanter: {EG.number_of_edges()}")
# Bort med eposter sendt til seg selv
EG.remove_edges_from(nx.selfloop_edges(EG))
isolerte = list(nx.isolates(EG)) # kan ikke bruke iteratorer direkte
EG.remove_nodes_from(isolerte)
print(f"Antall noder etter isolerte: {EG.number_of_nodes()}")
print(f"Antall nkanter etter isolerte: {EG.number_of_edges()}")

Antall noder: 57194
Andtall kanter: 103731
Antall noder etter isolerte: 57189
Antall nkanter etter isolerte: 103083



For å holde kompleksiteten nede koverterer vi til en urettet graf.

In [12]:
UG = EG.to_undirected(EG)
print(f"Antall noder etter konvertering: {UG.number_of_nodes()}")
print(f"Andtall kanter etter konvertering: {UG.number_of_edges()}")

Antall noder etter konvertering: 57189
Andtall kanter etter konvertering: 92442


Da bygger vi listen med noder, hvor antallet følger av antall kanter.  Det vil si at populære noder finnes ofte i listen.

In [ ]:
alle_noder = []
for u, v in EG.edges():
    # En kant gir to noder.
    alle_noder.extend([u,v])
#
print(f"Antall noder i listen: {len(alle_noder)}")

Antall noder i listen: 206166


Jeg spør Gemini:
```
Using networkx, without converting the whole graph to a list, how can I find a nrandom node
```
Den svarer
```
random_node = random.sample(G.nodes, 1)[0]
```
Men når jeg fortsetter:
```
Are you sure?  In your code "random_node = random.sample(G.nodes, 1)[0]" seems to create a list before returning one element
```
Svaret er
```
To be intellectually honest: internally, it still performs an $O(n)$ operation, [...]
```
Eller, som alltid: Livet er lettere når man vet svaret.

In [8]:
# Laste inn bakmann
Bakmann = nx.read_graphml("grafer/Bakmann.graphml")
# For å verifisere at vi finner de riktige nodene
nx.set_node_attributes(Bakmann, True, name="Bakmann")
# Sjekke at det ser bra ut  
print(f"Bakmann: kanter: {Bakmann.number_of_edges()}")
print(f"Bakmann: noder: {Bakmann.number_of_nodes()}")


Bakmann: kanter: 49
Bakmann: noder: 24


In [9]:
# Laste inn "money mule"
Esel = nx.read_graphml("grafer/Mule10.graphml")
# For å verifisere at vi finner de riktige nodene
nx.set_node_attributes(Esel, True, name="Esel")
# Verify by checking the number of nodes and edges
print(f"Nodes: {Esel.number_of_nodes()}")
print(f"Edges: {Esel.number_of_edges()}")
Esel_nodene = list(Esel.nodes())

Nodes: 44
Edges: 119


In [10]:
# Laste inn deling av utbytte
Utbytte = nx.read_graphml("grafer/Utbytte.graphml")
# For å verifisere at vi finner de riktige nodene
nx.set_node_attributes(Utbytte, True, name="Utbytte")
# Verify by checking the number of nodes and edges
print(f"Nodes: {Utbytte.number_of_nodes()}")
print(f"Edges: {Utbytte.number_of_edges()}")
Utbytte_nodene = list(Utbytte.nodes())

Nodes: 81
Edges: 159


In [ ]:
# Legg de små grafene til i den store
